# VIBE dataset vs a Gaussian with the same spectrum: robust-prune graphs

1. **Download** one dataset (`DATASET`) from [vector-index-bench/vibe](https://huggingface.co/datasets/vector-index-bench/vibe) into `DOWNLOAD_DIR`, using only the Python standard library (no `huggingface_hub`). The file streams into `<name>.hdf5.part`, an interrupted download resumes from where it stopped, and the finished file is checked against the size and SHA-256 that Hugging Face lists before it is renamed into place. A complete file already in `DOWNLOAD_DIR` is reused. The machine running this cell needs internet access (on a cluster, a login or data-transfer node if compute nodes have none).
2. **Spectrum**: the exact covariance spectrum and effective dimension $d_\mathrm{eff}$ (participation ratio) of the corpus (`train`), via `effdim.py`. A **synthetic Gaussian** dataset is built with the same number of points and covariance eigenvalues: $x = z \odot \sqrt{\lambda}$ with $z \sim \mathcal N(0, I)$. A robust-prune graph under Euclidean distance does not change under rotation or translation, so matching the eigenvalues is enough; the eigenvectors and mean are not needed.
3. **Robust prune** (`batchedEuclideanRobustPrune` from `utils.py`, `PRUNE_ALPHA = 1`: fully navigable) on both datasets. Graphs are written in the format of `simulrun.py`'s adjacency lists, one line per source: `source [neighbor1, neighbor2, ...]`, to `GRAPH_DIR/adj-list-<name>-alpha1-euclidean.txt`, with a `<name>-alpha1-euclidean-computed.txt` list of finished sources so an interrupted run resumes.
4. **Graph statistics** for both, read back from the stored graphs: out- and in-degrees, edges, reciprocity, next to the spectrum statistics.

**Distances.** The prune is Euclidean. Datasets with `distance = cosine` or `angular` are L2-normalised first (on unit vectors Euclidean distance orders points exactly as cosine distance does, so the graph is the cosine graph); `any` / `normalized` datasets are already unit-norm; `euclidean` is used as is. Inner-product (`ip`), `hamming` / `binary` and `chamfer` datasets are rejected.

**Size.** Robust prune scores every source against all `n` points, so the full graph costs about `n²` distance evaluations per dataset: use the GPU for datasets of ~1M points, or `MAX_POINTS` for a random subset. Both datasets are held in RAM as float32 (2 × n × d × 4 bytes, e.g. ~6 GB for 1M × 768).

In [ ]:
import hashlib
import json
import sys
import types
import urllib.request
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from effdim import accumulate, row_blocks, spectrum, spectrum_summary
from paths import RESULTS_DIR

REPO_ID = "vector-index-bench/vibe"
DATASET = "landmark-nomic-768-normalized"   # file name without .hdf5 (see the README tables in the repo)
DOWNLOAD_DIR = None      # where to save the .hdf5, e.g. "/scratch/<netid>/vibe"
GRAPH_DIR = RESULTS_DIR / "vibe_graphs"     # adjacency lists, spectra and the results table
VERIFY_SHA256 = True     # check the download's checksum (reads the file once more, ~seconds per GB)
MAX_POINTS = None        # None: the whole train set; int k: a seeded random subset of k points
SEED = 0
PRUNE_ALPHA = 1.0        # alpha-reachability; 1.0 = basic robust prune (fully navigable graph)
USE_GPU = None           # None: CuPy if it imports and sees a CUDA GPU, else NumPy
BATCH_SIZE = None        # None: 512 on GPU, 64 on CPU. GPU/RAM use ~ 5 * BATCH_SIZE * n * 4 bytes
RECIPROCITY = True       # also compute the share of edges u->v whose reverse v->u exists (~24 bytes per edge)

assert DOWNLOAD_DIR is not None, "set DOWNLOAD_DIR to where the dataset should be downloaded"
DOWNLOAD_DIR = Path(DOWNLOAD_DIR).expanduser()


def cuda_available():
    """True if the real CuPy imports and sees at least one CUDA device; prints why not otherwise."""
    try:
        import cupy
        if getattr(cupy, "_placeholder", False):
            print("GPU check: cupy in this kernel is the CPU-path placeholder; restart the kernel")
            return False
        n = cupy.cuda.runtime.getDeviceCount()
        if n == 0:
            print("GPU check: CuPy sees no CUDA devices")
        return n > 0
    except Exception as e:
        print(f"GPU check failed: {type(e).__name__}: {e}")
        return False


if USE_GPU is None:
    USE_GPU = cuda_available()
if BATCH_SIZE is None:
    BATCH_SIZE = 512 if USE_GPU else 64
print(f"USE_GPU = {USE_GPU}, BATCH_SIZE = {BATCH_SIZE}  (python: {sys.executable})")

## 1. Download

In [ ]:
def remote_file_info(filename):
    """(size in bytes, sha256 or None) of a file in the Hugging Face dataset repo."""
    api = f"https://huggingface.co/api/datasets/{REPO_ID}/tree/main"
    req = urllib.request.Request(api, headers={"User-Agent": "python-urllib"})
    with urllib.request.urlopen(req, timeout=60) as r:
        files = {f["path"]: f for f in json.load(r) if f.get("type") == "file"}
    if filename not in files:
        names = sorted(p.removesuffix(".hdf5") for p in files if p.endswith(".hdf5"))
        raise FileNotFoundError(f"{filename} is not in {REPO_ID}. Datasets: {', '.join(names)}")
    f = files[filename]
    return f["size"], (f.get("lfs") or {}).get("oid")


def sha256(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as f, tqdm(total=path.stat().st_size, unit="B", unit_scale=True, desc="sha256") as bar:
        while buf := f.read(chunk):
            h.update(buf)
            bar.update(len(buf))
    return h.hexdigest()


def download(filename, dest_dir, chunk=1 << 24):
    """Download `filename` from the repo into dest_dir, resuming a partial `.part` file; returns the path."""
    size, digest = remote_file_info(filename)
    dest = dest_dir / filename
    if dest.exists() and dest.stat().st_size == size:
        print(f"already downloaded: {dest}")
        return dest
    dest_dir.mkdir(parents=True, exist_ok=True)
    part = dest.with_name(filename + ".part")
    have = part.stat().st_size if part.exists() else 0
    if have > size:                                   # stale partial from a different file version
        part.unlink()
        have = 0
    if have < size:
        url = f"https://huggingface.co/datasets/{REPO_ID}/resolve/main/{filename}"
        headers = {"User-Agent": "python-urllib"} | ({"Range": f"bytes={have}-"} if have else {})
        with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=60) as r:
            if have and r.status != 206:              # server ignored the range: start over
                have = 0
            with open(part, "ab" if have else "wb") as f, \
                    tqdm(total=size, initial=have, unit="B", unit_scale=True, desc=filename) as bar:
                while buf := r.read(chunk):
                    f.write(buf)
                    bar.update(len(buf))
    if part.stat().st_size != size:
        raise IOError(f"{part}: {part.stat().st_size:,} of {size:,} bytes; rerun this cell to resume")
    if VERIFY_SHA256 and digest and sha256(part) != digest:
        part.unlink()
        raise IOError(f"{filename}: SHA-256 mismatch (partial file deleted); rerun this cell to download again")
    part.rename(dest)
    return dest


path = download(f"{DATASET}.hdf5", DOWNLOAD_DIR)

with h5py.File(path, "r") as f:
    attrs = {k: (v.decode() if isinstance(v, bytes) else v) for k, v in f.attrs.items()}
    contents = {k: (f[k].shape, str(f[k].dtype)) for k in f.keys()}
print(path, f"({path.stat().st_size / 1e9:.2f} GB)")
print("attributes:", attrs)
print("contents:  ", contents)

## 2. Spectrum and the synthetic Gaussian

`train` is loaded as float32 (and L2-normalised for cosine / angular). The covariance spectrum is exact: one pass of `effdim.accumulate` over all points.

In [ ]:
DISTANCE = str(attrs.get("distance", "")).lower()
POINT_TYPE = str(attrs.get("point_type", "float")).lower()
if DISTANCE not in {"euclidean", "cosine", "angular", "any", "normalized"} or POINT_TYPE == "binary":
    raise ValueError(f"{DATASET}: distance={DISTANCE!r}, point_type={POINT_TYPE!r}; "
                     "this notebook builds Euclidean robust-prune graphs (euclidean, cosine, any)")

with h5py.File(path, "r") as f:
    train = f["train"]
    n_all = train.shape[0]
    if MAX_POINTS is not None and MAX_POINTS < n_all:
        rows = np.sort(np.random.default_rng(SEED).choice(n_all, MAX_POINTS, replace=False))
        X = train[rows]                       # h5py reads sorted index lists
    else:
        X = train[...]
X = np.ascontiguousarray(X, dtype=np.float32)
if DISTANCE in ("cosine", "angular"):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X /= np.where(norms > 0, norms, 1)
N, D = X.shape
NAME = DATASET if N == n_all else f"{DATASET}-n{N}-seed{SEED}"     # graph and spectrum file names
print(f"{NAME}: {N:,} x {D} float32 ({X.nbytes / 1e9:.2f} GB), distance={DISTANCE}, point_type={POINT_TYPE}")

In [ ]:
acc = accumulate(row_blocks(X), N, NAME, device="numpy")
lam = spectrum(acc.covariance())                   # eigenvalues, largest first
d_eff = acc.value()
print(f"d_eff = {d_eff:.1f}  (d_eff / d = {d_eff / D:.3f})")

# Synthetic Gaussian: same n, same covariance eigenvalues.
X_syn = np.random.default_rng(SEED + 1).standard_normal((N, D), dtype=np.float32)
X_syn *= np.sqrt(lam).astype(np.float32)
acc_syn = accumulate(row_blocks(X_syn), N, "gaussian", device="numpy")
lam_syn = spectrum(acc_syn.covariance())
print(f"synthetic sample d_eff = {acc_syn.value():.1f}  (target {d_eff:.1f})")

GRAPH_DIR.mkdir(parents=True, exist_ok=True)
np.save(GRAPH_DIR / f"{NAME}-spectrum.npy", lam)

In [ ]:
INK, INK_2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
SERIES = ["#2a78d6", "#eb6834"]   # dataset, synthetic
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "axes.titlecolor": INK, "text.color": INK,
    "xtick.color": INK_2, "ytick.color": INK_2, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
})

# Eigenvalues in decreasing order, divided by the largest.
plt.figure(figsize=(8, 4))
k = np.arange(1, D + 1)
plt.plot(k, lam / lam[0], color=SERIES[0], lw=2, label=f"{DATASET}")
plt.plot(k, lam_syn / lam_syn[0], color=SERIES[1], lw=2, ls="--", label="Gaussian (sample spectrum)")
plt.xlim(1, D)
plt.ylim(0, 1)
plt.xlabel("rank k")
plt.ylabel("λ_k / λ_1")
plt.title(f"Covariance spectrum, d_eff = {d_eff:.1f} of d = {D}")
plt.legend(frameon=False)
plt.show()

## 3. Robust prune

`USE_GPU` picks `batchedEuclideanRobustPrune` (CuPy) or `batchedEuclideanRobustPruneCPU` (NumPy / BLAS). `utils.py` imports CuPy and numba at the top; on the CPU path, empty placeholder modules (marked `_placeholder`) are registered for whichever of them fails to import, so the NumPy functions can be imported.

In [ ]:
if USE_GPU:
    import cupy as cp
    if getattr(cp, "_placeholder", False):
        raise RuntimeError("this kernel holds the CPU-path placeholder for cupy from an earlier run: "
                           "restart the kernel, then run all cells")
    from utils import batchedEuclideanRobustPrune as prune, precomputeAugMatrix as augment
else:
    try:
        import cupy, cupyx.scipy.spatial.distance
    except Exception:
        for name in ("cupy", "cupyx", "cupyx.scipy", "cupyx.scipy.spatial", "cupyx.scipy.spatial.distance"):
            sys.modules[name] = types.ModuleType(name)
            sys.modules[name]._placeholder = True
        sys.modules["cupyx.scipy.spatial.distance"].cdist = None
    try:
        import numba
    except ImportError:
        numba = sys.modules["numba"] = types.ModuleType("numba")
        numba.njit, numba.prange, numba._placeholder = (lambda *a, **k: (lambda f: f)), range, True
    from utils import batchedEuclideanRobustPruneCPU as prune, precomputeAugMatrixCPU as augment
print(f"prune: {prune.__name__}  (batch size {BATCH_SIZE})")


def graph_paths(name):
    """Adjacency list and computed-sources files, named as in simulrun.py."""
    stem = f"{name}-alpha" + f"{PRUNE_ALPHA:g}".replace(".", "p") + "-euclidean"
    return GRAPH_DIR / f"adj-list-{stem}.txt", GRAPH_DIR / f"{stem}-computed.txt"


def build_graph(name, X):
    """Robust-prune graph of X, appended to the adjacency list one batch at a time; resumes from
    the computed-sources file. Returns the adjacency list path."""
    adj_path, computed_path = graph_paths(name)
    done = set(map(int, computed_path.read_text().split())) if computed_path.exists() else set()
    order = np.random.default_rng(SEED + 2).permutation(len(X))
    todo = [int(s) for s in order if int(s) not in done]
    print(f"{name}: {len(done):,} sources done, {len(todo):,} to go -> {adj_path}")
    if not todo:
        return adj_path

    data = cp.asarray(X) if USE_GPU else X
    X_aug = augment(data)
    for start in tqdm(range(0, len(todo), BATCH_SIZE), unit="batch", desc=name):
        batch = todo[start : start + BATCH_SIZE]
        neighborhoods = prune(batch, data, X_aug, alpha=PRUNE_ALPHA)
        # Whole batch at once; a crash loses at most one batch (computed is written after adj).
        with open(adj_path, "a") as adj, open(computed_path, "a") as comp:
            for source, hood in zip(batch, neighborhoods):
                adj.write(f"{source} {[v for v, _ in hood]}\n")
            comp.write("".join(f"{source}\n" for source in batch))
    del data, X_aug
    if USE_GPU:
        cp.get_default_memory_pool().free_all_blocks()
    return adj_path

In [ ]:
GRAPHS = {DATASET: build_graph(NAME, X), "gaussian": build_graph(f"{NAME}-gaussian", X_syn)}

## 4. Graph statistics

Read back from the stored adjacency lists. A source written twice (a run interrupted between writing a batch's adjacency lines and its computed list) keeps its last line. In-degree and reciprocity are reported only for complete graphs.

In [ ]:
def read_graph(adj_path):
    """{source: neighbor array} from an adjacency list file."""
    hoods = {}
    with open(adj_path) as f:
        for line in f:
            source, rest = line.split(" ", 1)
            hoods[int(source)] = np.fromstring(rest.strip()[1:-1], dtype=np.int64, sep=",")
    return hoods


def graph_stats(hoods, n):
    sources = np.fromiter(hoods, dtype=np.int64, count=len(hoods))
    out_deg = np.fromiter((len(h) for h in hoods.values()), dtype=np.int64, count=len(hoods))
    stats = {"sources": len(hoods), "edges": int(out_deg.sum()),
             "out mean": out_deg.mean(), "out median": np.median(out_deg),
             "out p99": np.percentile(out_deg, 99), "out max": out_deg.max()}
    if len(hoods) == n:
        dst = np.concatenate(list(hoods.values()))
        in_deg = np.bincount(dst, minlength=n)
        stats |= {"in median": np.median(in_deg), "in p99": np.percentile(in_deg, 99), "in max": in_deg.max(),
                  "in zero": int((in_deg == 0).sum())}
        if RECIPROCITY:
            src = np.repeat(sources, out_deg)
            stats["reciprocity"] = np.isin(dst * n + src, src * n + dst).mean()
    return stats, out_deg


rows, out_degrees = [], {}
for (label, adj_path), (lam_k, acc_k) in zip(GRAPHS.items(), [(lam, acc), (lam_syn, acc_syn)]):
    stats, out_degrees[label] = graph_stats(read_graph(adj_path), N)
    s = spectrum_summary(lam_k)
    rows.append({"dataset": label, "n": N, "d": D, "d_eff": acc_k.value(), "d_eff / d": acc_k.value() / D,
                 "dims for 50%": s["dims for 50%"], "dims for 90%": s["dims for 90%"],
                 "dims for 99%": s["dims for 99%"], **stats})
results = pd.DataFrame(rows)
results.to_csv(GRAPH_DIR / f"{NAME}-results.csv", index=False)
results.style.format({"n": "{:,}", "d_eff": "{:.1f}", "d_eff / d": "{:.3f}", "sources": "{:,}", "edges": "{:,}",
                      "out mean": "{:.1f}", "out median": "{:.0f}", "out p99": "{:.0f}", "out max": "{:.0f}",
                      "in median": "{:.0f}", "in p99": "{:.0f}", "in max": "{:.0f}", "in zero": "{:,.0f}",
                      "reciprocity": "{:.3f}"}, na_rep="–").hide(axis="index")

In [ ]:
# Out-degree distributions on shared bins.
plt.figure(figsize=(8, 4))
hi = max(x.max() for x in out_degrees.values())
bins = np.arange(0, hi + 2) if hi <= 200 else np.linspace(0, hi + 1, 101)
for color, (label, x) in zip(SERIES, out_degrees.items()):
    counts, edges = np.histogram(x, bins=bins)
    plt.stairs(counts / len(x), edges, color=color, lw=2, label=f"{label} (mean {x.mean():.1f})")
plt.xlabel("out-degree")
plt.ylabel("fraction of sources")
plt.title(f"Robust-prune out-degree, n = {N:,}")
plt.legend(frameon=False)
plt.show()